# Embedded AI Final Project

#### 20223265 조해영, 20214904 조성

0. 환경 설정

In [1]:
import json
import platform
import sys

import torch
import torchvision

# ROCm(AMD Radeon) 빌드의 PyTorch도 torch.cuda API를 그대로 쓰므로 device 문자열은 "cuda"이다.
if torch.version.hip:
    GPU_BACKEND = f"ROCm/HIP {torch.version.hip} (AMD)"
elif torch.version.cuda:
    GPU_BACKEND = f"CUDA {torch.version.cuda} (NVIDIA)"
else:
    GPU_BACKEND = "none (CPU-only PyTorch build)"

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("GPU backend:", GPU_BACKEND)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_props = torch.cuda.get_device_properties(0)
    gpu_name = gpu_props.name
    gpu_arch = getattr(gpu_props, "gcnArchName", None) or f"sm_{gpu_props.major}{gpu_props.minor}"
    print(f"GPU: {gpu_name} ({gpu_arch}, {gpu_props.total_memory / 1024**3:.1f} GiB)")
else:
    DEVICE = "cpu"
    gpu_name = gpu_arch = None
    print("[warn] No GPU detected; running on CPU.")
    if not torch.version.hip and not torch.version.cuda:
        print(
            "       This PyTorch build is CPU-only. For an AMD Radeon RX 6800 (gfx1030) on Windows:\n"
            '       pip install --index-url https://stable.repo.amd.com/rocm/whl-next/ '
            '"torch[device-gfx1030]" "torchvision[device-gfx1030]"'
        )

ENV_INFO = {
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "gpu_backend": GPU_BACKEND,
    "device": DEVICE,
    "gpu_name": gpu_name,
    "gpu_arch": gpu_arch,
    "platform": platform.platform(),
}
print()
print(json.dumps(ENV_INFO, indent=2, ensure_ascii=False))

python: 3.13.9
torch: 2.15.0+rocm10.2.0
torchvision: 0.29.2+rocm10.2.0
GPU backend: ROCm/HIP 7.17.26434 (AMD)
GPU available: True
GPU: AMD Radeon RX 6800 (gfx1030, 16.0 GiB)

{
  "python_version": "3.13.9",
  "torch_version": "2.15.0+rocm10.2.0",
  "torchvision_version": "0.29.2+rocm10.2.0",
  "gpu_backend": "ROCm/HIP 7.17.26434 (AMD)",
  "device": "cuda",
  "gpu_name": "AMD Radeon RX 6800",
  "gpu_arch": "gfx1030",
  "platform": "Windows-11-10.0.26200-SP0"
}


### 1. 모델 호출

Classification을 위한 ViT를 호출한다

In [2]:
import timm
from timm.data import create_transform, resolve_model_data_config

MODEL_NAME = "vit_base_patch16_224"
model = timm.create_model(
    MODEL_NAME, 
    num_classes=1000,
    pretrained=True).to(DEVICE)
model.eval()

data_config = resolve_model_data_config(model)
eval_transform = create_transform(**data_config, is_training=False)

print("model:", MODEL_NAME)
print("device:", DEVICE)
print("preprocessing:", data_config)

c:\Users\josun\Desktop\embeded_ai\Embedded-AI-Final-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model: vit_base_patch16_224
device: cuda
preprocessing: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'crop_pct': 0.9, 'crop_mode': 'center'}


In [3]:
total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
state_dict_size_bytes = sum(
    tensor.numel() * tensor.element_size() for tensor in model.state_dict().values()
)
state_dict_size_mib = state_dict_size_bytes / (1024 ** 2)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"State dict tensor size: {state_dict_size_mib:.2f} MiB ({state_dict_size_bytes:,} bytes)")

Total parameters: 86,567,656
Trainable parameters: 86,567,656
State dict tensor size: 330.23 MiB (346,270,624 bytes)


### ImageNet calibration / test 데이터

In [4]:
import shutil
import tarfile
from pathlib import Path

from huggingface_hub import hf_hub_download

# ImageNetV2 matched-frequency (10,000 images, ~1.26 GB) from the official Hugging Face mirror.
# The archive is cached under ./data/imagenetv2 (resumable) and extracted to ./ImageNetV2-matched-frequency.
IMAGENETV2_ROOT = Path("./ImageNetV2-matched-frequency")
IMAGENETV2_REPO = "vaishaal/ImageNetV2"
IMAGENETV2_ARCHIVE = "imagenetv2-matched-frequency.tar.gz"
IMAGENETV2_ARCHIVE_DIR = "imagenetv2-matched-frequency-format-val"  # top-level folder inside the archive
IMAGENETV2_SIZE = 10_000


def count_images(root: Path) -> int:
    return sum(1 for path in root.rglob("*") if path.suffix.lower() in {".jpeg", ".jpg", ".png"})


if IMAGENETV2_ROOT.is_dir() and count_images(IMAGENETV2_ROOT) == IMAGENETV2_SIZE:
    print(f"Using existing ImageNetV2 files: {IMAGENETV2_ROOT.resolve()}")
else:
    archive_path = Path(
        hf_hub_download(
            repo_id=IMAGENETV2_REPO,
            filename=IMAGENETV2_ARCHIVE,
            repo_type="dataset",
            local_dir="./data/imagenetv2",
        )
    )
    print(f"Archive: {archive_path} ({archive_path.stat().st_size / 1e9:.2f} GB)")

    extract_dir = Path("./data/imagenetv2/_extract")
    shutil.rmtree(extract_dir, ignore_errors=True)
    print("Extracting...")
    with tarfile.open(archive_path, "r:*") as archive:  # auto-detects gzip / plain tar
        archive.extractall(extract_dir, filter="data")

    shutil.rmtree(IMAGENETV2_ROOT, ignore_errors=True)
    shutil.move(extract_dir / IMAGENETV2_ARCHIVE_DIR, IMAGENETV2_ROOT)
    shutil.rmtree(extract_dir, ignore_errors=True)

    num_images = count_images(IMAGENETV2_ROOT)
    if num_images != IMAGENETV2_SIZE:
        raise RuntimeError(f"Expected {IMAGENETV2_SIZE} images, found {num_images} in {IMAGENETV2_ROOT}")
    print(f"ImageNetV2 ready: {IMAGENETV2_ROOT.resolve()} ({num_images} images)")

Using existing ImageNetV2 files: C:\Users\josun\Desktop\embeded_ai\Embedded-AI-Final-Project\ImageNetV2-matched-frequency


In [5]:
from pathlib import Path

from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder

# Local ImageNetV2 layout: ImageNetV2-matched-frequency/<class_id>/*.jpeg
IMAGENETV2_ROOT = Path("./ImageNetV2-matched-frequency")
CALIBRATION_RATIO = 0.2
BATCH_SIZE = 32
NUM_WORKERS = 0  # Reliable in notebooks on Windows.
SEED = 42
NUM_CLASSES = 1000

if not IMAGENETV2_ROOT.is_dir():
    raise FileNotFoundError(f"ImageNetV2 folder was not found: {IMAGENETV2_ROOT.resolve()}")


class NumericClassImageFolder(ImageFolder):
    def find_classes(self, directory):
        classes = [entry.name for entry in Path(directory).iterdir() if entry.is_dir()]
        if not all(class_name.isdigit() for class_name in classes):
            raise ValueError("ImageNetV2 class folders must be numeric IDs from 0 to 999.")
        classes.sort(key=int)
        expected_classes = [str(class_id) for class_id in range(NUM_CLASSES)]
        if classes != expected_classes:
            raise ValueError(
                f"Expected class folders 0-999, found {len(classes)} folders under {directory}."
            )
        return classes, {class_name: int(class_name) for class_name in classes}


dataset = NumericClassImageFolder(IMAGENETV2_ROOT, transform=eval_transform)
rng = torch.Generator().manual_seed(SEED)
class_indices = {class_id: [] for class_id in range(NUM_CLASSES)}
for index, class_id in enumerate(dataset.targets):
    class_indices[class_id].append(index)

calibration_indices = []
val_indices = []
calibration_counts = []
val_counts = []
for class_id, indices in class_indices.items():
    if len(indices) < 2:
        raise ValueError(f"Class {class_id} needs at least 2 images to split into both datasets.")
    shuffled = torch.randperm(len(indices), generator=rng).tolist()
    split_at = round(len(indices) * CALIBRATION_RATIO)
    split_at = min(max(split_at, 1), len(indices) - 1)
    calibration_indices.extend(indices[position] for position in shuffled[:split_at])
    val_indices.extend(indices[position] for position in shuffled[split_at:])
    calibration_counts.append(split_at)
    val_counts.append(len(indices) - split_at)

calibration_dataset = Subset(dataset, calibration_indices)
val_dataset = Subset(dataset, val_indices)

loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE == "cuda",
}
calibration_loader = DataLoader(calibration_dataset, shuffle=False, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)

print(f"Dataset: {IMAGENETV2_ROOT.resolve()}")
print(f"ImageNet classes: {len(dataset.classes)}")
print(f"Calibration images: {len(calibration_dataset)}")
print(f"Test images: {len(val_dataset)}")
print(f"Per-class calibration counts: {sorted(set(calibration_counts))}")
print(f"Per-class test counts: {sorted(set(val_counts))}")
print(f"Image transform: {eval_transform}")

Dataset: C:\Users\josun\Desktop\embeded_ai\Embedded-AI-Final-Project\ImageNetV2-matched-frequency
ImageNet classes: 1000
Calibration images: 2000
Test images: 8000
Per-class calibration counts: [2]
Per-class test counts: [8]
Image transform: Compose(
    Resize(size=248, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.5000, 0.5000, 0.5000]), std=tensor([0.5000, 0.5000, 0.5000]))
)


### 3. Baseline 모델 실행

In [6]:
model.eval()
correct = 0
total = 0

with torch.inference_mode():
    for images, labels in val_loader:
        images = images.to(DEVICE, non_blocking=DEVICE == "cuda")
        labels = labels.to(DEVICE, non_blocking=DEVICE == "cuda")

        predictions = model(images).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

if total == 0:
    raise ValueError("Validation set is empty.")

val_accuracy = correct / total
print(f"Validation accuracy: {val_accuracy:.2%} ({correct}/{total})")

Validation accuracy: 75.35% (6028/8000)
